In [114]:
import json
import requests
from bs4 import BeautifulSoup
import io
from pydub import AudioSegment
import os, re, unicodedata
import torch, torchaudio
import numpy as np
from pathlib import Path

In [25]:
book_ids = [
  {
    "book_id": "MAT",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
  },
  {
    "book_id": "MRK",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  },
  {
    "book_id": "LUK",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
  },
  {
    "book_id": "JHN",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
  },
  {
    "book_id": "ACT",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
  },
  {
    "book_id": "ROM",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  },
  {
    "book_id": "1CO",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  },
  {
    "book_id": "2CO",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
  },
  {
    "book_id": "GAL",
    "chapters": [1, 2, 3, 4, 5, 6]
  },
  {
    "book_id": "EPH",
    "chapters": [1, 2, 3, 4, 5, 6]
  },
  {
    "book_id": "PHP",
    "chapters": [1, 2, 3, 4]
  },
  {
    "book_id": "COL",
    "chapters": [1, 2, 3, 4]
  },
  {
    "book_id": "1TH",
    "chapters": [1, 2, 3, 4, 5]
  },
  {
    "book_id": "2TH",
    "chapters": [1, 2, 3]
  },
  {
    "book_id": "1TI",
    "chapters": [1, 2, 3, 4, 5, 6]
  },
  {
    "book_id": "2TI",
    "chapters": [1, 2, 3, 4]
  },
  {
    "book_id": "TIT",
    "chapters": [1, 2, 3]
  },
  {
    "book_id": "PHM",
    "chapters": [1]
  },
  {
    "book_id": "HEB",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
  },
  {
    "book_id": "JAS",
    "chapters": [1, 2, 3, 4, 5]
  },
  {
    "book_id": "1PE",
    "chapters": [1, 2, 3, 4, 5]
  },
  {
    "book_id": "2PE",
    "chapters": [1, 2, 3]
  },
  {
    "app_id": "1JN",
    "chapters": [1, 2, 3, 4, 5]
  },
  {
    "book_id": "2JN",
    "chapters": [1]
  },
  {
    "book_id": "3JN",
    "chapters": [1]
  },
  {
    "book_id": "JUD",
    "chapters": [1]
  },
  {
    "book_id": "REV",
    "chapters": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
  }
]

In [55]:
def fetch_next_data(url):
    next_data = None
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        script_tag = soup.find("script", id="__NEXT_DATA__")

        if script_tag and script_tag.string:
            next_data = json.loads(script_tag.string)
    return next_data

def extract_verse_text(next_data):
    versets = next_data["props"]["pageProps"]["chapterText"]
    data = {}
    for verset in versets:
        book_id = verset["book_id"]
        chapter_id = verset["chapter"]
        verse_id = verset["verse_start"]
        data[str(book_id) + "_" + str(chapter_id) + "_" + str(verse_id)] = verset["verse_text"].replace("\n ", "")
    return data

In [60]:
langues = ["YBBCAB"]
results = {}

for langue in langues:
    results[langue] = {}
    for book_id in book_ids:
        if 'book_id' not in book_id:
            continue
        book = book_id['book_id']
        results[langue][book] = {}
        for chapter in book_id["chapters"]:
            url = f"https://live.bible.is/bible/{langue}/{book}/{chapter}"
            next_data = fetch_next_data(url)
            results[langue][book][book + str("_") + str(chapter)] = extract_verse_text(next_data)

In [61]:
import json
with open("data_yemba/YBBCAB.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [73]:
def fetch_audio_link(langue, book, chapter):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    url = f"https://live.bible.is/api/bibles/filesets/{langue}N2DA?book_id={book}&chapter_id={chapter}&type=audio_drama"
    response = requests.get(url, headers=headers)
    audio_data = response.json()
    if 'data' in audio_data:
        if len(audio_data['data']) > 0:
            if 'path' in audio_data['data'][0]:
                audio_url = audio_data['data'][0]['path']
                response = requests.get(audio_url)
                if response.status_code in (200, 201, 204):
                    audio_stream = io.BytesIO(response.content)
                    sound = AudioSegment.from_file(audio_stream, format="mp3")
                    output_filename = f"data_yemba/audio_raw/{book}_{chapter}.wav"
                    sound.export(output_filename, format="wav")

In [74]:
for langue in langues:
    for book_id in book_ids:
        if 'book_id' not in book_id:
            continue
        book = book_id['book_id']
        for chapter in book_id["chapters"]:
            fetch_audio_link(langue, book, chapter)

In [89]:
with open("data_yemba/YBBCAB.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [103]:
data_ybbcab = data["YBBCAB"]
for book in book_ids:
    if 'book_id' not in book:
        continue
    book_id = book["book_id"]
    chapters = book["chapters"]
    book_dict = data_ybbcab[book_id]
    for chapter in chapters:
        versets = []
        verset_dict = book_dict[book_id + "_" + str(chapter)]
        error = False
        for v in range(1, len(verset_dict.keys()) + 1):
            try:
                verset_text = verset_dict[book_id + "_" + str(chapter) + "_" + str(v)]                
            except Exception as e:
                error = True
                break
            versets.append(verset_text)
        if not error:
            with open(f"data_yemba/versets_raw/{book_id}_{chapter}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(versets))
    

In [ ]:
OUT = "data_yemba/data"
os.makedirs(OUT, exist_ok=True)

# --- Normalisation : romanisation approximative pour MMS (a-z et ') ---
MAP = str.maketrans({"ɛ": "e", "ɔ": "o", "ə": "e", "ɨ": "i", "ʉ": "u",
                     "ŋ": "n", "ɓ": "b", "ɗ": "d", "ʼ": "'", "’": "'",
                     "ɑ": "a", "ʔ": "'"})

def normalize(word):
    w = word.lower().translate(MAP)
    w = unicodedata.normalize("NFKD", w)
    w = "".join(c for c in w if not unicodedata.combining(c))  # enlève les tons
    return re.sub(r"[^a-z']", "", w)

dossier = Path("data_yemba/versets_raw")
for chemin_fichier in dossier.iterdir():
    if chemin_fichier.is_file():
        TEXTE = str(chemin_fichier).replace("\\", "/").split('.')[0].strip()
        chapter_name = TEXTE.split("/")[-1]
        AUDIO = "data_yemba/audio_raw/" + chapter_name + ".wav"

        if not os.path.exists(AUDIO):
            continue

        # --- : liste de versets -> liste de mots ---
        versets = [l.strip() for l in open(TEXTE + ".txt", encoding="utf-8") if l.strip()]
        words, verse_of_word = [], []
        for vi, v in enumerate(versets):
            for w in v.split():
                n = normalize(w)
                if n:
                    words.append(n)
                    verse_of_word.append(vi)

        # --- Alignement MMS ---
        device = "cuda" if torch.cuda.is_available() else "cpu"
        bundle = torchaudio.pipelines.MMS_FA
        model = bundle.get_model(with_star=False).to(device)
        tokenizer, aligner = bundle.get_tokenizer(), bundle.get_aligner()

        # Lecture via pydub : mono + 16 kHz (fréquence attendue par MMS_FA)
        a16 = AudioSegment.from_wav(AUDIO).set_channels(1).set_frame_rate(bundle.sample_rate)
        samples = np.array(a16.get_array_of_samples(), dtype=np.float32)
        samples /= float(1 << (8 * a16.sample_width - 1))   # normalise dans [-1, 1]
        wav = torch.from_numpy(samples).unsqueeze(0)          # forme (1, nb_échantillons)

        with torch.inference_mode():
            emission, _ = model(wav.to(device))

        spans = aligner(emission[0], tokenizer(words))
        sec_per_frame = wav.size(1) / emission.size(1) / bundle.sample_rate
        w_start = [s[0].start * sec_per_frame for s in spans]
        w_end = [s[-1].end * sec_per_frame for s in spans]

        # --- Frontières : milieu du silence entre le dernier mot du verset i et le premier du i+1 ---
        audio = AudioSegment.from_wav(AUDIO)
        total = len(audio)
        bounds = [0]
        for vi in range(len(versets) - 1):
            last = max(i for i, v in enumerate(verse_of_word) if v == vi)
            nxt = min(i for i, v in enumerate(verse_of_word) if v == vi + 1)
            bounds.append(int((w_end[last] + w_start[nxt]) / 2 * 1000))
        bounds.append(total)

        for i in range(len(versets)): 
            seg = audio[bounds[i]:bounds[i + 1]]
            verset_name = chapter_name + "_" + str(i+1) + ".wav"
            name = f"{OUT}/{verset_name}"
            seg.export(name, format="wav")
            print(name)

data_yemba/data/1CO_1_1.wav
data_yemba/data/1CO_1_2.wav
data_yemba/data/1CO_1_3.wav
data_yemba/data/1CO_1_4.wav
data_yemba/data/1CO_1_5.wav
data_yemba/data/1CO_1_6.wav
data_yemba/data/1CO_1_7.wav
data_yemba/data/1CO_1_8.wav
data_yemba/data/1CO_1_9.wav
data_yemba/data/1CO_1_10.wav
data_yemba/data/1CO_1_11.wav
data_yemba/data/1CO_1_12.wav
data_yemba/data/1CO_1_13.wav
data_yemba/data/1CO_1_14.wav
data_yemba/data/1CO_1_15.wav
data_yemba/data/1CO_1_16.wav
data_yemba/data/1CO_1_17.wav
data_yemba/data/1CO_1_18.wav
data_yemba/data/1CO_1_19.wav
data_yemba/data/1CO_1_20.wav
data_yemba/data/1CO_1_21.wav
data_yemba/data/1CO_1_22.wav
data_yemba/data/1CO_1_23.wav
data_yemba/data/1CO_1_24.wav
data_yemba/data/1CO_1_25.wav
data_yemba/data/1CO_1_26.wav
data_yemba/data/1CO_1_27.wav
data_yemba/data/1CO_1_28.wav
data_yemba/data/1CO_1_29.wav
data_yemba/data/1CO_1_30.wav
data_yemba/data/1CO_1_31.wav
data_yemba/data/1CO_10_1.wav
data_yemba/data/1CO_10_2.wav
data_yemba/data/1CO_10_3.wav
data_yemba/data/1CO_10_

KeyboardInterrupt: 

In [122]:
with open("data_yemba/YBBCAB.json", "r", encoding="utf-8") as f:
    data = json.load(f)
yemba_data = data["YBBCAB"]

In [127]:
i = 1
results = []

dossier = Path("data_yemba/data")
for chemin_fichier in dossier.iterdir():
    if chemin_fichier.is_file():
        path = str(chemin_fichier).replace("\\", "/").split('.')[0].strip()
        verset_name = path.split("/")[-1]
        book_id = verset_name.split("_")[0]
        chapter_id = verset_name.split("_")[1]
        verset_id = verset_name.split("_")[2]
        if book_id in yemba_data:
            if book_id + "_" + chapter_id in yemba_data[book_id]:
                if book_id + "_" + chapter_id + "_" + verset_id in yemba_data[book_id][book_id + "_" + chapter_id]:
                    verset_text = yemba_data[book_id][book_id + "_" + chapter_id][book_id + "_" + chapter_id + "_" + verset_id]
                    results.append(
                        {
                            "id": f"sample_yem_{i:03d}", 
                            "audio_path": "data/audio/" + verset_name + ".wav",
                            "text": verset_text, 
                            "language_id": "yem"
                        }
                    )
                    i += 1

In [128]:
output_file = "data_yemba/my_data_train.jsonl"
lignes_json = [json.dumps(item, ensure_ascii=False) for item in results]
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(lignes_json))